# Model 6 = XGBoost

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import random

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)


In [3]:
customer_data = pd.read_excel('../Datasets/cleaned_customers_churn_data.xlsx')
x = customer_data.drop(columns=['Exited'])
y = customer_data['Exited']

In [4]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((8000, 11), (2000, 11), (8000,), (2000,))

In [23]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Model
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
# Train
xgb_model.fit(x_train, y_train)

# Predictions
y_pred_train = xgb_model.predict(x_train)
y_pred_test = xgb_model.predict(x_test)

# Accuracy
print("Training Accuracy:", accuracy_score(y_train, y_pred_train))
print("Testing Accuracy :", accuracy_score(y_test, y_pred_test))

# Confusion Matrix
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_test))

# Classification Report
print("\nClassification Report")
print(classification_report(y_test, y_pred_test))

# ROC-AUC
print("\nROC-AUC Score:", roc_auc_score(y_test, y_pred_test))

Training Accuracy: 0.885625
Testing Accuracy : 0.867

Confusion Matrix
[[1537   70]
 [ 196  197]]

Classification Report
              precision    recall  f1-score   support

           0       0.89      0.96      0.92      1607
           1       0.74      0.50      0.60       393

    accuracy                           0.87      2000
   macro avg       0.81      0.73      0.76      2000
weighted avg       0.86      0.87      0.86      2000


ROC-AUC Score: 0.7288564185631881


In [24]:
feature_importance = pd.DataFrame({
    'Feature': x.columns,
    'Importance': xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print(feature_importance)

            Feature  Importance
5     NumOfProducts    0.276434
7    IsActiveMember    0.195770
2               Age    0.161961
9           Germany    0.103945
4           Balance    0.059195
1            Gender    0.051138
10            Spain    0.036796
8   EstimatedSalary    0.030231
0       CreditScore    0.029511
3            Tenure    0.028038
6         HasCrCard    0.026980


In [25]:
import pickle

pickle.dump(xgb_model, open("../Models/churn_model.pkl", "wb"))